# Analysis SITCOM-1948: LSSTCam

SITCOM-1884 Rotator Torque Analysis with Pancake Wrap did not show big differences in the rotator torques before/after the installation of the pancake wrap on the top-end integration assembly. 

We want to dive deeper and perform an analysis “per movement”. Similar to SITCOM-1946: CamHex Torque "per movement" Analysis with Pancake Wrap, we want to convert the torque profile into metrics that we can use for statistical analysis. 

The idea is that a “movement” is defined by the time between `mtrotator.command_move` command and the `mtrotator.logevent_inPosition`. This will give us several data points per test period.

This notebook will focus on the LSSTCam data, taken on Level 3 Rotator testing. For the analysis of the ComCam CCW data, refer to the second notebook from this ticket. 

In [ ]:
#%matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

from astropy.time import Time
from pathlib import Path
from datetime import datetime
from scipy.integrate import simpson

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

# Set global font size for labels, titles, and ticks
plt.rcParams.update({
    "font.size": 14,  
    "axes.labelsize": 16,  
    "axes.titlesize": 18,  
    "xtick.labelsize": 14,  
    "ytick.labelsize": 14,  
})

efd_client = makeEfdClient()

In [ ]:
unit = r'N$\cdot$m'

# Data Analysis

For the LSSTCam you can choose the date:

Rotator Soak Tests were proceeded in the following time windows:
- From <code>2025-02-25 17:34 - 19:00</code>: first full range rotation tests without hexapod movements.
- From <code>2025-02-27 13:00 - 19:00</code>: LVV-T2261 - rotator and camera hexapod movements. 
- From <code>2025-02-27 23:00 - 2025-02-28 02:00</code>: BLOCK-T358 - small soak test using rotator and camera hexapod.
- From <code>2025-02-28 00:01 - 01:38 UTC</code>
- From <code>2025-02-28 17:38 - 17:49 UTC</code>
- From <code>2025-02-28 17:57 - 18:02 UTC</code>
- From <code>2025-03-01 13:35 - 15:54 UTC</code>

## First time analysis from 2025-02-25

In [ ]:
start_time_d1 = "2025-02-25T17:34:00"
end_time_d1 = "2025-02-25T19:00:00"

In [ ]:
start_time= Time(start_time_d1, scale="utc", format="isot")
end_time = Time(end_time_d1, scale="utc", format="isot")

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0","torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
move_rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.command_move",
    columns=["private_sndStamp", "position"],
    begin=start_time,
    end=end_time
)

In [ ]:
in_position = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.logevent_inPosition",
    columns=["private_sndStamp", "inPosition"],
    begin=start_time,
    end=end_time
)

In [ ]:
ccw = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTMount.cameraCableWrap",
    columns=["actualTorquePercentage0", "actualTorquePercentage1", "actualPosition"],
    begin=start_time,
    end=end_time
)

In [ ]:
move_rot["private_sndStamp"] = pd.to_datetime(move_rot["private_sndStamp"], unit="s").dt.tz_localize("UTC")
in_position["private_sndStamp"] = pd.to_datetime(in_position["private_sndStamp"], unit="s").dt.tz_localize("UTC")

# Subtract 37 seconds from each timestamp
move_rot["private_sndStamp"] = move_rot["private_sndStamp"] - pd.Timedelta(seconds=37)
in_position["private_sndStamp"] = in_position["private_sndStamp"] - pd.Timedelta(seconds=37)

Here we will plot the `move_command` (green dots) and `in_position` stamps (orange line) with in_position false and true (blue cross)

For easier reading the plot the suggestion is to uncomment the `%matplotlib widget` line in the first cell and zoom in into the specific plot range.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.scatter(
    move_rot["private_sndStamp"], move_rot["position"], 
    label="Move Command (Position)", color="darkcyan", s=30, alpha=0.7, zorder=2
)
ax1.set_ylabel("Position (deg)", color="darkcyan")
ax1.set_xlabel("Time")
plt.xticks(rotation=45)
ax1.tick_params(axis="y", labelcolor="darkcyan")

ax2 = ax1.twinx()

ax2.step(
    in_position["private_sndStamp"], in_position["inPosition"], 
    label="In Position", color="orange", where="post", linestyle="-", linewidth=2, zorder=2
)

ax2.plot(
    in_position["private_sndStamp"], in_position["inPosition"], 
    "x", color="navy", markersize=8, markeredgewidth=2, label="In Position (start/end)", zorder=5
)

ax2.set_ylabel("In Position (True/False)", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")

plt.title("LSSTCam: Rotator Move Command and In Position Status Over Time")

plt.show()

# Rotator position and torques for the whole time range

Let's plot the rotator position and the torques for the whole range of movement

In [ ]:
dayobs="2025-02-25"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))

max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]

line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
plt.xticks(rotation=45)
ax.set_ylabel(r"Torque (N$\cdot$m)")
ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"LSSTCam: Rotator Torque Analysis with Pancake Wrap {dayobs}: Rotator only")
ax.grid(which="major", axis="both", linestyle="--", alpha=0.25)
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)

plt.show()

Make a datafame for "per movement"

In [ ]:
move_times = move_rot["private_sndStamp"].values  # time when movement command is sent
in_position_times = in_position["private_sndStamp"].values  # time when position is reached

movement_intervals = zip(move_times, in_position_times)

print(f"Number of command_move events: {len(move_times)}")
print(f"Number of inPosition events: {len(in_position_times)}")

In [ ]:
move_times = pd.to_datetime(move_rot["private_sndStamp"].values, unit='s', utc=True)
in_position_times = pd.to_datetime(in_position["private_sndStamp"].values, unit='s', utc=True)

here we'll make the movenment from move command to in position=True

In [ ]:
movement_data = []

num_movements = len(move_times)

for i in range(num_movements):
    move_start = move_times[i]  # Start with the move command

    # Find the first occurrence of inPosition = True after move_start
    move_end = next(
        (row.private_sndStamp for _, row in in_position.iterrows() if row.inPosition == True and row.private_sndStamp > move_start),
        None
    )

    if move_end is None:
        move_end = move_start + pd.Timedelta(seconds=60)  
    
    time_diff = (move_end - move_start).total_seconds()
    print(f"Movement {i+1}: Move Start = {move_start}, Move End = {move_end}, Time Difference = {time_diff} seconds")

    mask_motor = (rot_motor.index >= move_start) & (rot_motor.index <= move_end)
    mask_rot = (rot.index >= move_start) & (rot.index <= move_end)

    torque0 = rot_motor.loc[mask_motor, "torque0"]
    torque1 = rot_motor.loc[mask_motor, "torque1"]
    position = rot.loc[mask_rot, "actualPosition"]
    time = rot_motor.loc[mask_motor].index  

    if not torque0.empty and not torque1.empty and not position.empty:
        movement_data.append({
            "movement_id": i + 1,
            "move_start": move_start,
            "move_end": move_end,
            "torque0": torque0.values,
            "torque1": torque1.values,
            "actualPosition": position.values,
            "time": time
        })

movement_df = pd.DataFrame(movement_data)

print(movement_df)
print(movement_df.head())


# Plotting the movements 

Let's plot all the movements

In [ ]:
if len(movement_df) > 0:
    for i, row in movement_df.iterrows():
        time = row['time']
        torque0 = row['torque0']
        torque1 = row['torque1']
        position = row['actualPosition']

        move_start = row['move_start']
        move_end = row['move_end']

        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(time, torque0, color="C0", label="Torque 0")
        ax.plot(time, torque1, color="C1", label="Torque 1")

        ax.axvline(move_start, color="gray", linestyle="--", label="Move Start")
        ax.axvline(move_end, color="black", linestyle="--", label="Move End")

        in_position_times = in_position[in_position["inPosition"] == True]["private_sndStamp"]
        for t in in_position_times:
            if move_start <= t <= move_end:
                ax.axvline(t, color="red", linestyle="dotted", alpha=0.7, label="In Position True")

        ax.set_xlabel("Time [UTC]")
        plt.xticks(rotation=45)
        ax.set_ylabel("Torque (N·m)")
        ax.set_title(f"LSSTCam: Torque vs Time for Movement {i + 1}")

        ax2 = ax.twinx()
        ax2.plot(time, position, color="C2", label="Position", linestyle="--")
        ax2.set_ylabel("Rotator Position (deg)")

        ax.xaxis.set_minor_locator(mdates.MicrosecondLocator(interval=500000))  # Minor ticks every 500ms
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f'))  # Show milliseconds

        plt.xticks(rotation=30, ha="right")

        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles + handles2, labels + labels2, loc="upper left")

        ax.grid(which="both", axis="both", linestyle="--", alpha=0.25)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

else:
    print("No valid movement pairs found.")


# Plotting unbiased movements 

Let's remove the biases calculate the areas under the curve and add it to the data frame

In [ ]:
if len(movement_df) > 0:
    areas = []  

    for i, row in movement_df.iterrows():
        time = row["time"]
        torque0 = row["torque0"]
        torque1 = row["torque1"]
        position = row["actualPosition"]

        # Remove bias by subtracting the mean
        torque0 -= np.mean(torque0)
        torque1 -= np.mean(torque1)

        # Compute the area under the absolute torque curves
        area_torque0 = simpson(np.abs(torque0), dx=1)  
        area_torque1 = simpson(np.abs(torque1), dx=1)

        total_area = area_torque0 + area_torque1
        areas.append(total_area)  

        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(time, torque0, color="C0", label="Torque 0 (Bias Removed)")
        ax.plot(time, torque1, color="C1", label="Torque 1 (Bias Removed)")

        ax.set_xlabel("Time [UTC]")
        plt.xticks(rotation=45)
        ax.set_ylabel("Torque (N·m)")
        ax.set_title(f"LSSTCam: Torque vs Time for Movement {i + 1}")

        ax2 = ax.twinx()
        ax2.plot(time, position, color="C2", label="Position", linestyle="--")
        ax2.set_ylabel("Rotator Position (deg)")

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.xaxis.set_major_locator(mdates.MinuteLocator())

        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles + handles2, labels + labels2, loc="upper left")

        ax.grid(which="major", axis="both", linestyle="--", alpha=0.25)

        area_text = f"Total Area: {total_area:.2f} N·m·s"
        ax.text(0.95, 0.05, area_text, transform=ax.transAxes,
                ha="right", va="bottom", fontsize=12, color="black",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", boxstyle="round,pad=0.5"))

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    movement_df["area (N m s)"] = areas

else:
    print("No valid movement pairs found.")


Let's calculate the movement size in degrees in rotator position

In [ ]:
movement_sizes = []

for i, row in movement_df.iterrows():
    position = row['actualPosition']
    
    movement_size = np.abs(position.max() - position.min())
    movement_sizes.append(movement_size)

movement_df['movement_size (degrees)'] = movement_sizes

print(movement_df[['movement_size (degrees)']])


# Area under the curve depending on movement size

Let's plot the area VS movement size

In [ ]:
if 'area (N m s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    plt.figure(figsize=(8, 6))
    plt.scatter(movement_df['movement_size (degrees)'], movement_df['area (N m s)'], color='C0', alpha=0.6)

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (N·m·s)')
    plt.title('LSSTCam: Area vs Movement Size for the rotator')

    plt.grid(":", alpha=0.25)

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


# Area versus Movement size: linear and poli fit 

Let's try to add some fit

In [ ]:
if 'area (N m s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    x = movement_df['movement_size (degrees)']
    y = movement_df['area (N m s)']
    
    # Perform linear regression (best-fit line)
    coefficients = np.polyfit(x, y, 1)  # 1 indicates a linear fit
    poly = np.poly1d(coefficients)

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, color='C0', alpha=0.6, label='Data points')
    
    plt.plot(x, poly(x), color='red', linestyle='--', label='Best fit line')

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (N·m·s)')
    plt.title('LSSTCam: Area vs Movement Size for the rotator')

    slope = coefficients[0]
    intercept = coefficients[1]
    equation = f'Fit: y = {slope:.6f}x + {intercept:.6f}'

    plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', color='black')

    plt.grid(":", alpha=0.25)

    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


Let's try some polinom fit

In [ ]:
if 'area (N m s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    x = movement_df['movement_size (degrees)']
    y = movement_df['area (N m s)']
    
    # Perform a polynomial fit (degree 3)
    coefficients = np.polyfit(x, y, 3)  # 3 indicates a cubic (degree 3) fit
    poly = np.poly1d(coefficients)

    x_fit = np.linspace(x.min(), x.max(), 500)  
    y_fit = poly(x_fit)

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, color='C0', alpha=0.6, label='Data points')

    plt.plot(x_fit, y_fit, color='red', linestyle='--', label='Polynomial fit (degree 3)')

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (N·m·s)')
    plt.title('LSSTCam: Area vs Movement Size with Polynomial Fit (Degree 3) - Rotator')

    equation = f'Fit: y = {coefficients[0]:.2e}x³ + {coefficients[1]:.2e}x² + {coefficients[2]:.2e}x + {coefficients[3]:.2e}'

    plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', color='black')

    plt.grid(":", alpha=0.25)

    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


# Plotting one movement

Let's plot only one movement of interest

In [ ]:
movement_index = 18  # index of the movement 

if len(movement_df) > movement_index:
    row = movement_df.iloc[movement_index]  
    
    time = row['time']
    torque0 = row['torque0']
    torque1 = row['torque1']
    position = row['actualPosition']

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(time, torque0, color="C0", label="Torque 0")
    ax.plot(time, torque1, color="C1", label="Torque 1")

    ax.set_xlabel("Time [UTC]")
    plt.xticks(rotation=45)
    ax.set_ylabel("Torque (N·m)")
    ax.set_title(f"LSSTCam: Torque vs Time for Movement {movement_index + 1}")

    ax2 = ax.twinx()
    ax2.plot(time, position, color="C2", label="Position", linestyle="--")
    ax2.set_ylabel("Rotator Position (deg)")

    ax.xaxis.set_major_locator(mdates.SecondLocator(interval=5)) 
    ax.xaxis.set_minor_locator(mdates.MicrosecondLocator(interval=500000))  
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  

    plt.xticks(rotation=90, ha="right")

    handles, labels = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(handles + handles2, labels + labels2, loc="upper left")

    ax.grid(which="major", axis="both", linestyle="--", alpha=0.25)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

else:
    print("Movement index out of range.")


# Here we will try some metrics, like Efficiency ratio: 

This metric tells you how much torque effort (area under the curve) is needed per degree of rotator movement.

Eq: ER=area/movenment size

In [ ]:
movement_df["efficiency_ratio"] = movement_df["area (N m s)"] / movement_df["movement_size (degrees)"]

print(movement_df["efficiency_ratio"].describe())

Let's do the histogram. If the values cluster around a specific range, it means most movements require similar torque per degree.

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(movement_df["efficiency_ratio"], bins=15, color="C0", alpha=0.7, edgecolor="black")
plt.xlabel("Efficiency Ratio (N·m·s per degree)")
plt.ylabel("Frequency")
plt.title("LSSTCam: Distribution of Efficiency Ratios for the rotator")
plt.grid(axis="y", linestyle="--", alpha=0.25)
plt.show()

ER VS Movement size. If we see a downward trend, it means larger movements are more efficient (less torque per degree).

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(movement_df["movement_size (degrees)"], movement_df["efficiency_ratio"], color="C0", alpha=0.7)
plt.xlabel("Movement Size (degrees)")
plt.ylabel("Efficiency Ratio (N·m·s per degree)")
plt.title("LSSTCam: Efficiency Ratio vs Movement Size for the rotator")
plt.grid(True, linestyle="--", alpha=0.25)
plt.show()

## CCW analysis

Here, we will repeat the analysis for LSSTCam but with the CCW data

In [ ]:
movement_ccw_data = []

num_movements = len(move_times)

for i in range(num_movements):
    move_start = move_times[i]  # Start with the move command

    # Find the first occurrence of inPosition = True after move_start
    move_end = next(
        (row.private_sndStamp for _, row in in_position.iterrows() if row.inPosition == True and row.private_sndStamp > move_start),
        None
    )

    if move_end is None:
        move_end = move_start + pd.Timedelta(seconds=60)  
    
    time_diff = (move_end - move_start).total_seconds()
    print(f"Movement {i+1}: Move Start = {move_start}, Move End = {move_end}, Time Difference = {time_diff} seconds")

    mask_ccw = (ccw.index >= move_start) & (ccw.index <= move_end)

    torque0 = ccw.loc[mask_ccw, "actualTorquePercentage0"]
    torque1 = ccw.loc[mask_ccw, "actualTorquePercentage1"]
    position = ccw.loc[mask_ccw, "actualPosition"]
    time = ccw.loc[mask_ccw].index  

    if not torque0.empty and not torque1.empty and not position.empty:
        movement_ccw_data.append({
            "movement_id": i + 1,
            "move_start": move_start,
            "move_end": move_end,
            "torque0": torque0.values,
            "torque1": torque1.values,
            "actualPosition": position.values,
            "time": time
        })

movement_ccw_df = pd.DataFrame(movement_ccw_data)

print(movement_ccw_df)
print(movement_ccw_df.head())


# Plotting the movement of the CCW

In [ ]:
if len(movement_ccw_df) > 0:
    for i, row in movement_ccw_df.iterrows():
        time = row['time']
        torque0 = row['torque0']
        torque1 = row['torque1']
        position = row['actualPosition']

        move_start = row['move_start']
        move_end = row['move_end']

        fig, ax = plt.subplots(figsize=(10, 5))

        # Plot torques
        ax.plot(time, torque0, color="C0", label="Torque 0")
        ax.plot(time, torque1, color="C1", label="Torque 1")

        # Highlight start and end of movement
        ax.axvline(move_start, color="gray", linestyle="--", label="Move Start")
        ax.axvline(move_end, color="black", linestyle="--", label="Move End")

        # Find and plot inPosition = True timestamps
        in_position_times = in_position[in_position["inPosition"] == True]["private_sndStamp"]
        for t in in_position_times:
            if move_start <= t <= move_end:
                ax.axvline(t, color="red", linestyle="-", alpha=0.7, label="In Position True")

        ax.set_xlabel("Time [UTC]")
        plt.xticks(rotation=45)
        ax.set_ylabel("Torque (%)")
        ax.set_title(f"LSSTCam: Torque vs Time for Movement {i + 1}")

        # Secondary y-axis for position
        ax2 = ax.twinx()
        ax2.plot(time, position, color="C2", label="Position", linestyle="--")
        ax2.set_ylabel("CCW Position (deg)")

#        ax.xaxis.set_major_locator(mdates.SecondLocator(interval=1))  # Major ticks every second
        ax.xaxis.set_minor_locator(mdates.MicrosecondLocator(interval=500000))  # Minor ticks every 500ms
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f'))  # Show milliseconds

        # Rotate x-axis labels for better readability
        plt.xticks(rotation=30, ha="right")

        # Combine legends
        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles + handles2, labels + labels2, loc="upper left")

        ax.grid(which="both", axis="both", linestyle="--", alpha=0.25)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

else:
    print("No valid movement pairs found.")


# Plotting the unbiased movement

Let's unbiased the data and calculate the area under the torques curves

In [ ]:
if len(movement_ccw_df) > 0:
    areas = []  

    for i, row in movement_ccw_df.iterrows():
        time = row["time"]
        torque0 = row["torque0"]
        torque1 = row["torque1"]
        position = row["actualPosition"]

        # Remove bias by subtracting the mean
        torque0 -= np.mean(torque0)
        torque1 -= np.mean(torque1)

        # Compute the area under the absolute torque curves
        area_torque0 = simpson(np.abs(torque0), dx=1)  
        area_torque1 = simpson(np.abs(torque1), dx=1)

        total_area = area_torque0 + area_torque1
        areas.append(total_area)  

        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(time, torque0, color="C0", label="Torque 0 (Bias Removed)")
        ax.plot(time, torque1, color="C1", label="Torque 1 (Bias Removed)")

        ax.set_xlabel("Time [UTC]")
        plt.xticks(rotation=45)
        ax.set_ylabel("Torque (%)")
        ax.set_title(f"LSSTCam: Torque vs Time for Movement {i + 1}")

        ax2 = ax.twinx()
        ax2.plot(time, position, color="C2", label="Position", linestyle="--")
        ax2.set_ylabel("CCW Position (deg)")

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.xaxis.set_major_locator(mdates.MinuteLocator())

        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles + handles2, labels + labels2, loc="upper left")

        ax.grid(which="major", axis="both", linestyle="--", alpha=0.25)

        area_text = f"Total Area: {total_area:.2f} %·s"
        ax.text(0.95, 0.05, area_text, transform=ax.transAxes,
                ha="right", va="bottom", fontsize=12, color="black",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", boxstyle="round,pad=0.5"))

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    movement_ccw_df["area (% x s)"] = areas

else:
    print("No valid movement pairs found.")


Calculating the movement size

In [ ]:
movement_sizes = []

for i, row in movement_ccw_df.iterrows():
    position = row['actualPosition']
    
    movement_size = np.abs(position.max() - position.min())
    movement_sizes.append(movement_size)

movement_ccw_df['movement_size (degrees)'] = movement_sizes

print(movement_ccw_df[['movement_size (degrees)']])


# Plot the area vs movement size

In [ ]:
if 'area (% x s)' in movement_ccw_df.columns and 'movement_size (degrees)' in movement_ccw_df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist2d(movement_ccw_df['movement_size (degrees)'], movement_ccw_df['area (% x s)'], bins=30, cmap='Blues')
    
    ax.set_xlabel('Movement Size (degrees)')
    ax.set_ylabel('Area (% x s)')
    ax.set_title('LSSTCam: Histogram of Area vs Movement Size for the CCW')
    
    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not found in the DataFrame.")


# Fitting the area versus movement size

In [ ]:
if 'area (% x s)' in movement_ccw_df.columns and 'movement_size (degrees)' in movement_ccw_df.columns:
    x = movement_ccw_df['movement_size (degrees)']
    y = movement_ccw_df['area (% x s)']
    
    # Perform linear regression (best-fit line)
    coefficients = np.polyfit(x, y, 1)  # 1 indicates a linear fit
    poly = np.poly1d(coefficients)

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, color='C0', alpha=0.6, label='Data points')
    
    plt.plot(x, poly(x), color='red', linestyle='--', label='Best fit line')

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (% x s)')
    plt.title('LSSTCam: Area vs Movement Size for the CCW')

    slope = coefficients[0]
    intercept = coefficients[1]
    equation = f'Fit: y = {slope:.6f}x + {intercept:.6f}'

    plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', color='black')

    plt.grid(":", alpha=0.25)

    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


# Effitiency ratio

In [ ]:
movement_ccw_df["efficiency_ratio"] = movement_ccw_df["area (% x s)"] / movement_ccw_df["movement_size (degrees)"]

print(movement_ccw_df["efficiency_ratio"].describe())

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(movement_ccw_df["efficiency_ratio"], bins=15, color="C0", alpha=0.7, edgecolor="black")
plt.xlabel("Efficiency Ratio (%·s per degree)")
plt.ylabel("Frequency")
plt.title("LSSTCam: Distribution of Efficiency Ratios for the CCW")
plt.grid(axis="y", linestyle="--", alpha=0.25)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(movement_ccw_df["movement_size (degrees)"], movement_ccw_df["efficiency_ratio"], color="C0", alpha=0.7)
plt.xlabel("Movement Size (degrees)")
plt.ylabel("Efficiency Ratio (%·s per degree)")
plt.title("LSSTCam: Efficiency Ratio vs Movement Size for the CCW")
plt.grid(True, linestyle="--", alpha=0.25)
plt.show()